In [1]:
import pandas as pd
import re
from collections import Counter
import os


# Language detection
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator

import medspacy


# NLP / Medical NLP
import spacy
import scispacy

I0000 00:00:1787843573.919188   29748 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787843574.219580   29748 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787843579.782214   29748 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787843579.785464   29748 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
W0000 00:0

In [2]:
df = pd.read_csv("../../data/train.csv")
df.head()

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df['Report']

0       Técnica: RMN de la rodilla. Resultados: Rotura...
1       [DATE]: * MR Knie Rechts 15ch AA Klinische Inl...
2       Hallazgos:\nNo hay alteraciones en significati...
3        In the medial compartment, the meniscus is no...
4       CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...
                              ...                        
4402    Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...
4403    [DATE]: *MR Knie Rechts 15ch AA Klinische Inli...
4404    SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...
4405    MRI of Knee with \n-Locator, SG PD FatSat, SG ...
4406    Técnica: RMN de la rodilla. Resultados: Rotura...
Name: Report, Length: 4407, dtype: object

In [4]:
def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"

    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

In [5]:
df["language"] = df["Report"].apply(detect_language)

In [6]:
df["language"].value_counts()


language
en    1736
es     682
tr     546
hr     406
el     321
de     262
bg     220
nl     153
fr      81
Name: count, dtype: int64

In [7]:
df[["Report", "language"]].sample(8)

,Report,language
2766,"In the medial compartment, there is bucket-han...",en
475,"SOL DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...",tr
2594,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr
3296,CONSTATATIONS :\n\nFractures :\nFracture sous-...,fr
3780,Lateral tibyal plato posteriyorundaki inkomple...,tr
4003,Diz eklemindeki trikompartmantal dejeneratif e...,tr
3403,Mediales Kompartiment:\nRegelrechtes Knochenma...,de
2096,Technique: MRI of the knee. ACL normal. MCL no...,en


In [8]:
excluded_terms = {"DATE"}

In [9]:
def find_abbreviations(text):
    pattern = r'\b[A-Z]{2,}(?:[-/][A-Z0-9]+)*\b'
    
    terms = re.findall(pattern, text)
    
    return [
        term for term in terms
        if term not in excluded_terms
    ]

In [10]:
text = df["Report"].iloc[1]

print(text)
print(find_abbreviations(text))

[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindingen:
['MR', 'AA', 'DRB']


In [11]:
all_abbreviations = Counter()

for text in df["Report"].dropna():
    all_abbreviations.update(find_abbreviations(text))

In [12]:
all_abbreviations.most_common(20)

[('ACL', 1729),
 ('MRI', 1526),
 ('MCL', 1447),
 ('PCL', 1303),
 ('COMPARTMENT', 1076),
 ('LCL', 940),
 ('LIGAMENTS', 694),
 ('FS', 693),
 ('II', 681),
 ('FINDINGS', 556),
 ('OA', 467),
 ('KNEE', 445),
 ('TT-TG', 440),
 ('IMPRESSION', 430),
 ('MPFL', 395),
 ('MR', 382),
 ('TECHNIQUE', 378),
 ('MRG', 370),
 ('LATERAL', 370),
 ('MEDIAL', 368)]

In [13]:

# MEDICAL DICTIONARY


medical_dictionary = {

    
    # LIGAMENTS
    

    "anterior cruciate ligament": "ACL",
    "anterior cruciate ligament (ACL)": "ACL",
    "ACL": "ACL",
    "anterior cruciate ligament injury": "ACL injury",
    "ACL injury": "ACL injury",
    "ACL sprain": "ACL sprain",
    "grade 1 injury of ACL": "grade 1 ACL injury",
    "grade 1 ACL injury": "grade 1 ACL injury",
    "grade 1 injury or degeneration of ACL": "grade 1 ACL injury or degeneration",
    "partial tear of ACL": "partial ACL tear",
    "ACL partial tear": "partial ACL tear",
    "tear of ACL": "ACL tear",
    "ACL tear": "ACL tear",
    "subtotal ACL tear": "subtotal ACL tear",
    "ACL mucinous degeneration": "ACL mucoid degeneration",
    "ACL myxoid degeneration": "ACL mucoid degeneration",
    "ACL mucoid degeneration": "ACL mucoid degeneration",

    "πρόσθιος χιαστός σύνδεσμος": "ACL",
    "πρόσθιου χιαστού συνδέσμου": "ACL",
    "πρόσθιος χιαστός": "ACL",
    "πρόσθιου χιαστού": "ACL",

    "ön çapraz bağ": "ACL",
    "ön çapraz bağda": "ACL",
    "ön çapraz bağın": "ACL",

    "posterior cruciate ligament": "PCL",
    "posterior cruciate ligament (PCL)": "PCL",
    "PCL": "PCL",
    "PCL injury": "PCL injury",
    "PCL sprain": "PCL sprain",
    "grade 1 injury of PCL": "grade 1 PCL injury",
    "grade I injury of PCL": "grade 1 PCL injury",
    "grade 1 PCL injury": "grade 1 PCL injury",
    "partial tear of PCL": "partial PCL tear",
    "partial tear of the PCL": "partial PCL tear",
    "PCL partial tear": "partial PCL tear",
    "severe partial tear of PCL": "severe partial PCL tear",
    "PCL avulsion fracture": "PCL avulsion fracture",
    "avulsion fracture of PCL": "PCL avulsion fracture",

    "οπίσθιος χιαστός σύνδεσμος": "PCL",
    "οπίσθιου χιαστού συνδέσμου": "PCL",
    "οπίσθιος χιαστός": "PCL",
    "οπίσθιου χιαστού": "PCL",

    "arka çapraz bağ": "PCL",
    "arka çapraz bağda": "PCL",

    "medial collateral ligament": "MCL",
    "medial collateral ligament (MCL)": "MCL",
    "MCL": "MCL",
    "MCL sprain": "MCL sprain",
    "grade I sprain of MCL": "grade 1 MCL sprain",
    "grade II sprain of MCL": "grade 2 MCL sprain",
    "partial tear of MCL": "partial MCL tear",
    "MCL partial tear": "partial MCL tear",

    "ligamento colateral medial": "MCL",
    "ligamento colateral interno": "MCL",
    "LCM": "MCL",
    "lesión LCM": "MCL injury",

    "medial kollateral ligaman": "MCL",
    "medial kollateral ligamanında": "MCL",
    "medial kollateral ligaman çevresinde": "MCL",

    "lateral collateral ligament": "LCL",
    "lateral collateral ligament (LCL)": "LCL",
    "LCL": "LCL",


    
    # MENISCI
    

    "medial meniscus": "medial meniscus",
    "medial meniscus tear": "medial meniscus tear",
    "tear of medial meniscus": "medial meniscus tear",
    "posterior horn of medial meniscus": "posterior horn of medial meniscus",
    "posterior horn of medial meniscus tear": "posterior horn medial meniscus tear",
    "medial meniscus posterior horn tear": "posterior horn medial meniscus tear",
    "medial meniscus degeneration": "medial meniscus degeneration",
    "grade 2 medial meniscus degeneration": "grade 2 medial meniscus degeneration",
    "grade 3 medial meniscus degeneration": "grade 3 medial meniscus degeneration",
    "medial meniscus extrusion": "medial meniscus extrusion",
    "medial meniscus degeneration with extrusion": "medial meniscus degeneration with extrusion",

    "lateral meniscus": "lateral meniscus",
    "lateral meniscus tear": "lateral meniscus tear",
    "tear of lateral meniscus": "lateral meniscus tear",
    "anterior horn of lateral meniscus": "anterior horn of lateral meniscus",
    "anterior horn of lateral meniscus tear": "anterior horn lateral meniscus tear",
    "posterior horn of lateral meniscus": "posterior horn of lateral meniscus",
    "lateral meniscus degeneration": "lateral meniscus degeneration",
    "grade 1 lateral meniscus degeneration": "grade 1 lateral meniscus degeneration",
    "grade 2 lateral meniscus degeneration": "grade 2 lateral meniscus degeneration",

    "meniscal tear": "meniscus tear",
    "meniscus tear": "meniscus tear",
    "meniscal degeneration": "meniscus degeneration",

    "grade 2 intrasubstance signal":
        "grade 2 meniscal intrasubstance degeneration",

    "grade 2 intrasubstance signal of posterior horn of medial meniscus":
        "grade 2 medial meniscus degeneration",

    "meniscopathy": "meniscus degeneration",
    "medial meniscopathy": "medial meniscus degeneration",
    "grade 3 meniscopathy": "grade 3 medial meniscus degeneration",

    "medial menisküs": "medial meniscus",
    "medial menisküste": "medial meniscus",
    "lateral menisküs": "lateral meniscus",
    "lateral menisküste": "lateral meniscus",
    "meniskopati": "meniscus degeneration",
    "medial meniskopati": "medial meniscus degeneration",

    "medijalni menisk": "medial meniscus",
    "medijalnog meniska": "medial meniscus",
    "lateralni menisk": "lateral meniscus",
    "lateralnog meniska": "lateral meniscus",
    "ruptura medijalnog meniska": "medial meniscus tear",
    "degeneracija medijalnog meniska": "medial meniscus degeneration",


    
    # FRACTURES / BONE INJURIES
    

    "tibial plateau fracture": "tibial plateau fracture",
    "fracture of tibial plateau": "tibial plateau fracture",
    "split fracture of tibial plateau": "split tibial plateau fracture",

    "tibial intercondylar eminence fracture":
        "tibial intercondylar eminence fracture",

    "intercondylar eminence fracture":
        "tibial intercondylar eminence fracture",

    "posterior tibial intercondylar eminence fracture":
        "posterior tibial intercondylar eminence fracture",

    "bone contusion": "bone contusion",
    "bone bruise": "bone contusion",
    "bone marrow edema": "bone marrow edema",
    "bone marrow oedema": "bone marrow edema",
    "bone edema": "bone marrow edema",
    "osteochondral lesion": "osteochondral lesion",
    "osteochondral injury": "osteochondral injury",
    "Segond fracture": "Segond fracture",
    "Segond's fracture": "Segond fracture",
    "osteonecrosis": "osteonecrosis",
    "small osteonecrosis": "small osteonecrosis",


    
    # CARTILAGE / OSTEOARTHRITIS
    

    "osteoarthritis": "osteoarthritis",
    "osteoarthritis of knee": "knee osteoarthritis",
    "knee osteoarthritis": "knee osteoarthritis",
    "gonarthrosis": "knee osteoarthritis",
    "gonartroz": "knee osteoarthritis",

    "chondromalacia": "chondromalacia",
    "grade 1 chondromalacia": "grade 1 chondromalacia",
    "grade 2 chondromalacia": "grade 2 chondromalacia",
    "grade 3 chondromalacia": "grade 3 chondromalacia",
    "grade 4 chondromalacia": "grade 4 chondromalacia",
    "grade 3-4 chondromalacia": "grade 3-4 chondromalacia",
    "patellar chondromalacia": "patellar chondromalacia",
    "patellofemoral chondromalacia": "patellofemoral chondromalacia",
    "cartilage degeneration": "cartilage degeneration",
    "cartilage thinning": "cartilage thinning",
    "full-thickness cartilage defect": "full-thickness cartilage defect",

    "hondromalazi": "chondromalacia",
    "hondromalacia": "chondromalacia",
    "hondromalacija": "chondromalacia",


    
    # SYNOVIUM / FLUID / BURSA
    

    "synovitis": "synovitis",
    "knee synovitis": "knee synovitis",
    "joint effusion": "joint effusion",
    "knee joint effusion": "knee joint effusion",
    "minimal joint effusion": "minimal joint effusion",
    "mild joint effusion": "mild joint effusion",
    "suprapatellar bursitis": "suprapatellar bursitis",
    "deep infrapatellar bursitis": "deep infrapatellar bursitis",
    "infrapatellar bursitis": "infrapatellar bursitis",
    "anserine bursitis": "anserine bursitis",
    "pes anserine bursitis": "pes anserine bursitis",
    "patellar plica": "patellar plica",
    "synovial hypertrophy": "synovial hypertrophy",
    "lipoma arborescens": "lipoma arborescens",
    "lipohemarthrosis": "lipohemarthrosis",


    
    # CYSTS
    

    "Baker's cyst": "Baker's cyst",
    "Baker cyst": "Baker's cyst",
    "popliteal cyst": "popliteal cyst",
    "popliteal cystic lesion": "popliteal cyst",
    "ganglion cyst": "ganglion cyst",
    "ACL ganglion cyst": "ACL ganglion cyst",
    "PCL ganglion cyst": "PCL ganglion cyst",
    "ligament ganglion cyst": "ligament ganglion cyst",
    "parameniscal cyst": "parameniscal cyst",
    "parameniscal cystic lesion": "parameniscal cyst",
    "synovial cyst": "synovial cyst",


    
    # TENDONS
    

    "patellar tendinosis": "patellar tendinosis",
    "patellar tendon tendinosis": "patellar tendinosis",
    "quadriceps tendinosis": "quadriceps tendinosis",
    "quadriceps tendon tendinosis": "quadriceps tendinosis",
    "patellar tendon injury": "patellar tendon injury",
    "quadriceps tendon injury": "quadriceps tendon injury",
    "tendinopathy": "tendinopathy",
    "tendinosis": "tendinosis",
    "grade 1 tendon injury": "grade 1 tendon injury",

    # Variants for post-translation normalization
    "rupture of fibers of the patellar tendon":
        "patellar tendon rupture",

    "rupture of the patellar tendon":
        "patellar tendon rupture",

    "patellar tendon rupture":
        "patellar tendon rupture",


    
    # SOFT TISSUE
    

    "soft tissue edema": "soft tissue edema",
    "subcutaneous edema": "subcutaneous edema",
    "muscle edema": "muscle edema",

    "subcutaneous and muscle edema":
        "subcutaneous and muscle edema",

    "hematoma": "hematoma",
    "subcutaneous hematoma": "subcutaneous hematoma",
    "prepatellar hematoma": "prepatellar hematoma",


    
    # PATELLA / PATELLOFEMORAL
    

    "patellar maltracking": "patellar maltracking",
    "patella maltracking": "patellar maltracking",
    "patellar subluxation": "patellar subluxation",
    "patellar dislocation": "patellar dislocation",
    "patella alta": "patella alta",
    "Wiberg type I patella": "Wiberg type I patella",
    "Wiberg type II patella": "Wiberg type II patella",
    "Wiberg II": "Wiberg type II patella",
    "bipartite patella": "bipartite patella",


    
    # OTHER LESIONS
    

    "osteochondroma": "osteochondroma",
    "exostosis": "exostosis",
    "osteochondral loose body": "osteochondral loose body",
    "loose body": "loose body",
    "osteochondral foreign body": "osteochondral loose body",
    "intraosseous hemangioma": "intraosseous hemangioma",
    "nodular fasciitis": "nodular fasciitis",

    "tenosynovial giant cell tumor":
        "tenosynovial giant cell tumor",

    "fibroma of the tendon sheath":
        "fibroma of the tendon sheath",

    "tenosynovial fibroma":
        "tenosynovial fibroma",


    
    # ANATOMICAL / OTHER RECURRING CONDITIONS
    

    "IT band syndrome": "iliotibial band syndrome",
    "iliotibial band syndrome": "iliotibial band syndrome",
    "iliotibial band impingement": "iliotibial band impingement",
    "pes anserine paratendinitis": "pes anserine paratendinitis",
    "patellar tendon impingement": "patellar tendon impingement",
    "patellofemoral syndrome": "patellofemoral syndrome",


    
    # MRI
    

    "magnetic resonance imaging": "MRI",
    "magnetic resonance imaging (MRI)": "MRI",
    "MRI": "MRI",
    "resonancia magnética": "MRI",
    "resonancia magnética nuclear": "MRI",

    # RM queda fuera deliberadamente por su ambigüedad.


    
    # MPFL
    

    "medial patellofemoral ligament": "MPFL",
    "medial patellofemoral ligament (MPFL)": "MPFL",
    "MPFL": "MPFL",
    "MPFL injury": "MPFL injury",
    "MPFL tear": "MPFL tear",
    "partial tear of MPFL": "partial MPFL tear",
    "MPFL partial tear": "partial MPFL tear",
    "grade 1 injury of MPFL": "grade 1 MPFL injury",
    "grade 1 MPFL injury": "grade 1 MPFL injury",


    
    # OA
    

    "OA": "osteoarthritis",


    
    # FS
    

    # Se detecta pero NO se reemplaza automáticamente.
    "FS": "FS",
}



# TÉRMINOS QUE REQUIEREN REVISIÓN


FLAG_FOR_REVIEW = {"FS"}



# SIGLAS CORTAS


SHORT_ACRONYMS = {
    "ACL",
    "PCL",
    "MCL",
    "LCL",
    "MRI",
    "MPFL",
    "OA",
    "FS"
}



# PROTECCIÓN DE TÉRMINOS


def protect_terms(text, glossary):
    """
    Protege términos médicos antes de enviarlos al traductor.

    Las frases más largas se procesan primero para evitar colisiones.

    Las siglas se detectan con coincidencia exacta y respetando
    mayúsculas/minúsculas.

    Los términos largos se detectan ignorando diferencias
    de mayúsculas/minúsculas.

    Se utilizan placeholders alfanuméricos:
        XTERM0X
        XTERM1X
        XTERM2X
    """

    placeholders = {}
    flagged_terms = []

    
    # FRASES LARGAS PRIMERO
    

    sorted_terms = sorted(
        glossary.keys(),
        key=len,
        reverse=True
    )

    for i, term in enumerate(sorted_terms):

        target = glossary[term]

        
        # PATRÓN DE MATCHING
        

        if term in SHORT_ACRONYMS:

            pattern = re.compile(
                r'(?<!\w)' + re.escape(term) + r'(?!\w)'
            )

        else:

            pattern = re.compile(
                r'(?<!\w)' + re.escape(term) + r'(?!\w)',
                re.IGNORECASE
            )

        
        # FLAGGED TERMS
        

        if term in FLAG_FOR_REVIEW:

            if pattern.search(text):
                flagged_terms.append(term)

            # NO reemplazar
            continue

        
        # PROTEGER TÉRMINO
        

        if pattern.search(text):

            token = f"XTERM{i}X"

            text = pattern.sub(token, text)

            placeholders[token] = target

    return text, placeholders, flagged_terms



# RESTAURAR TÉRMINOS


def restore_terms(translated_text, placeholders):
    """
    Restaura los términos médicos después de la traducción.

    Tolera modificaciones menores que pueda introducir
    el traductor:

        XTERM0X
        XTERM 0X
        XTERM0 X
        xterm0x
        X TERM 0 X
    """

    for token, term in placeholders.items():

        # Extraer número:
        # XTERM123X -> 123
        token_number = re.search(r'\d+', token)

        if not token_number:
            continue

        number = token_number.group()

        # Patrón flexible para posibles alteraciones
        # del traductor.
        pattern = re.compile(
            r'x\s*term\s*' +
            re.escape(number) +
            r'\s*x',
            re.IGNORECASE
        )

        translated_text = pattern.sub(
            term,
            translated_text
        )

    return translated_text


# TRANSLATION VALIDATION


def validate_translation(
    translated_text,
    restored_text,
    placeholders
):
    """
    Performs basic validation after translation and restoration.

    The goal is to detect obvious translation/restoration
    problems, not to guarantee medical correctness.
    """

    warnings = []

    
    # 1. Check that translated text is not empty
    

    if not translated_text or not translated_text.strip():

        warnings.append(
            "Translation returned empty text."
        )

    
    # 2. Check that restored text is not empty
    

    if not restored_text or not restored_text.strip():

        warnings.append(
            "Restored text is empty."
        )

    
    # 3. Check for remaining placeholders
    

    remaining_placeholders = re.findall(
        r'x\s*term\s*\d+\s*x',
        restored_text,
        re.IGNORECASE
    )

    if remaining_placeholders:

        warnings.append(
            "Unrestored placeholders detected: "
            + str(remaining_placeholders)
        )

    
    # 4. Check that protected medical terms were restored
    

    for token, medical_term in placeholders.items():

        if medical_term not in restored_text:

            warnings.append(
                f"Medical term may have been lost: "
                f"{medical_term}"
            )

    
    # Final result
    

    is_valid = len(warnings) == 0

    return {
        "is_valid": is_valid,
        "warnings": warnings
    }

# MEDICAL TERM NORMALIZATION


def normalize_medical_terms(text, glossary):
    """
    Normaliza términos médicos después de la traducción.

    Esta etapa se utiliza para detectar variantes que el traductor
    puede generar aunque no hayan sido protegidas originalmente.

    Las frases más largas se procesan primero para evitar
    reemplazos parciales.
    """

    # Procesar primero las expresiones más largas
    sorted_terms = sorted(
        glossary.keys(),
        key=len,
        reverse=True
    )

    for term in sorted_terms:

        normalized_term = glossary[term]

        # Si ya está normalizado, no hacer nada
        if term.lower() == normalized_term.lower():
            continue

        # Coincidencia case-insensitive
        pattern = re.compile(
            r'(?<!\w)' +
            re.escape(term) +
            r'(?!\w)',
            re.IGNORECASE
        )

        text = pattern.sub(
            normalized_term,
            text
        )

    return text



# EJEMPLO DE USO


if __name__ == "__main__":

    sample = (
        "Se observa lesión LCM y ruptura medijalnog meniska, "
        "con OA leve y hallazgo de FS dudoso. "
        "Se recomienda MRI de control. "
        "Grade 3 meniscopathy also noted."
    )

    
    # PROTEGER
    

    protected_text, placeholders, flagged = protect_terms(
        sample,
        medical_dictionary
    )

    print("\n--- TEXTO PROTEGIDO ---")
    print(protected_text)

    print("\n--- PLACEHOLDERS ---")
    print(placeholders)

    print("\n--- FLAGGED ---")
    print(flagged)

    
    # AQUÍ IRÍA GOOGLE TRANSLATE
    

    fake_translated = protected_text

    
    # RESTAURAR
    

    final_text = restore_terms(
        fake_translated,
        placeholders
    )

    print("\n--- TEXTO RESTAURADO ---")
    print(final_text)

    
    # NORMALIZAR
    

    normalized_text = normalize_medical_terms(
        final_text,
        medical_dictionary
    )

    print("\n--- TEXTO NORMALIZADO ---")
    print(normalized_text)


--- TEXTO PROTEGIDO ---
Se observa XTERM203X y XTERM55X, con XTERM222X leve y hallazgo de FS dudoso. Se recomienda XTERM221X de control. XTERM107X also noted.

--- PLACEHOLDERS ---
{'XTERM55X': 'medial meniscus tear', 'XTERM107X': 'grade 3 medial meniscus degeneration', 'XTERM203X': 'MCL injury', 'XTERM221X': 'MRI', 'XTERM222X': 'osteoarthritis'}

--- FLAGGED ---
['FS']

--- TEXTO RESTAURADO ---
Se observa MCL injury y medial meniscus tear, con osteoarthritis leve y hallazgo de FS dudoso. Se recomienda MRI de control. grade 3 medial meniscus degeneration also noted.

--- TEXTO NORMALIZADO ---
Se observa MCL injury y medial meniscus tear, con osteoarthritis leve y hallazgo de FS dudoso. Se recomienda MRI de control. grade 3 medial meniscus degeneration also noted.


In [14]:

# MEDICAL CONTENT VALIDATION


def validate_medical_content(text):
    """
    Checks for basic medical consistency after translation.
    """

    warnings = []

    # Check important anatomical structures
    important_terms = [
        "ACL",
        "PCL",
        "MCL",
        "LCL",
        "medial meniscus",
        "lateral meniscus",
        "patellar tendon",
        "quadriceps tendon",
        "joint effusion",
        "synovitis",
        "Baker's cyst",
        "fracture",
        "bone contusion",
        "osteoarthritis"
    ]

    # Check for suspicious translation artifacts
    suspicious_patterns = [
        "XTERM",
        "undefined",
        "null",
        "None"
    ]

    for pattern in suspicious_patterns:

        if pattern.lower() in text.lower():

            warnings.append(
                f"Suspicious text detected: {pattern}"
            )

    # Empty text
    if not text or not text.strip():

        warnings.append(
            "Medical text is empty."
        )

    return {
        "is_valid": len(warnings) == 0,
        "warnings": warnings
    }

In [15]:

# DATASET LABELS


LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

In [16]:
# NLP LABEL EXTRACTION

def extract_nlp_labels(text):
    """
    Converts the normalized medical report
    into the 12 dataset labels.
    """

    labels = {label: 0 for label in LABELS}

    return labels

In [17]:

# MEDICAL FINDINGS → DATASET LABELS


LABEL_PATTERNS = {

    "ACL": [
        r"\bACL\b",
        r"anterior cruciate ligament",
        r"ACL tear",
        r"ACL injury",
        r"ACL sprain",
    ],

    "MCL": [
        r"\bMCL\b",
        r"medial collateral ligament",
        r"MCL tear",
        r"MCL injury",
        r"MCL sprain",
    ],

    "Medial Meniscus": [
        r"medial meniscus",
        r"medial meniscus tear",
        r"medial meniscus degeneration",
        r"medial meniscopathy",
    ],

    "Lateral Meniscus": [
        r"lateral meniscus",
        r"lateral meniscus tear",
        r"lateral meniscus degeneration",
    ],

    "Medial OA": [
        r"medial compartment osteoarthritis",
        r"medial compartment OA",
        r"medial femorotibial osteoarthritis",
    ],

    "Lateral OA": [
        r"lateral compartment osteoarthritis",
        r"lateral compartment OA",
        r"lateral femorotibial osteoarthritis",
    ],

    "PF OA": [
        r"patellofemoral osteoarthritis",
        r"patellofemoral OA",
        r"patellofemoral chondromalacia",
        r"patellar chondromalacia",
    ],

    "Effusion": [
        r"joint effusion",
        r"knee joint effusion",
        r"effusion",
        r"hemarthrosis",
    ],

    "Synovitis": [
        r"synovitis",
        r"synovial hypertrophy",
    ],

    "Baker's": [
        r"Baker's cyst",
        r"Baker cyst",
        r"popliteal cyst",
    ],

    "Contusion": [
        r"bone contusion",
        r"bone bruise",
        r"muscle contusion",
        r"contusion",
    ],

    "Fracture": [
        r"fracture",
        r"tibial plateau fracture",
        r"osteochondral fracture",
        r"insufficiency fracture",
    ]
}

In [18]:
def translate_to_english(text, source_language):
    """
    Translates text into English using Google Translate.

    Parameters
    ----------
    text : str
        Text to translate.

    source_language : str
        ISO 639-1 language code detected from the source text.
        Examples:
            es = Spanish
            en = English
            tr = Turkish
            hr = Croatian
            el = Greek

    Returns
    -------
    str
        Translated text in English.
    """

    translator = GoogleTranslator(
        source=source_language,
        target="en"
    )

    return translator.translate(text)

In [20]:
# 1. Detect source language
source_language = detect_language(text)

# 2. Protect medical terms
protected_text, placeholders, flagged_terms = protect_terms(
    text,
    medical_dictionary
)

# 3. Translate protected text into English
translated_text = translate_to_english(
    protected_text,
    source_language
)

# 4. Restore medical terms
restored_text = restore_terms(
    translated_text,
    placeholders
)

print("\n--- RESTORED TEXT ---")
print(restored_text)



# 5. TRANSLATION VALIDATION


validation = validate_translation(
    translated_text,
    restored_text,
    placeholders
)

print("\n--- TRANSLATION VALIDATION ---")
print("Valid:", validation["is_valid"])

if validation["warnings"]:

    print("Warnings:")

    for warning in validation["warnings"]:
        print("-", warning)



# 6. MEDICAL TERM NORMALIZATION


normalized_text = normalize_medical_terms(
    restored_text,
    medical_dictionary
)

print("\n--- NORMALIZED TEXT ---")
print(normalized_text)


--- RESTORED TEXT ---
Technique: MRI of the knee. Results: Rupture of fibers of the patellar tendon. Impression: Rupture of fibers of the patellar tendon.

--- TRANSLATION VALIDATION ---
Valid: True

--- NORMALIZED TEXT ---
Technique: MRI of the knee. Results: patellar tendon rupture. Impression: patellar tendon rupture.


In [21]:
nlp = spacy.load("en_core_sci_sm")

In [22]:
text_for_scispacy = normalized_text
doc = nlp(text_for_scispacy)

In [23]:
token_data = [
    {
        "text": token.text,
        "lemma": token.lemma_,
        "pos": token.pos_,
        "dep": token.dep_
    }
    for token in doc
]

lemmatized_tokens = [
    token["lemma"]
    for token in token_data
    if token["pos"] != "PUNCT"
]

print(lemmatized_tokens)

['technique', 'mri', 'of', 'the', 'knee', 'result', 'patellar', 'tendon', 'rupture', 'impression', 'patellar', 'tendon', 'rupture']


In [24]:
medical_terms_found = []

for term, normalized_term in medical_dictionary.items():

    pattern = re.compile(
        r'(?<!\w)' + re.escape(term) + r'(?!\w)',
        re.IGNORECASE
    )

    for match in pattern.finditer(normalized_text):

        medical_terms_found.append({
            "found": match.group(),
            "normalized": normalized_term,
            "start": match.start(),
            "end": match.end()
        })

print(medical_terms_found)

[{'found': 'patellar tendon rupture', 'normalized': 'patellar tendon rupture', 'start': 37, 'end': 60}, {'found': 'patellar tendon rupture', 'normalized': 'patellar tendon rupture', 'start': 74, 'end': 97}, {'found': 'MRI', 'normalized': 'MRI', 'start': 11, 'end': 14}]


In [25]:
def get_section(text, position):

    before_text = text[:position].lower()

    if "impression:" in before_text:
        return "Impression"

    elif "results:" in before_text:
        return "Results"

    elif "technique:" in before_text:
        return "Technique"

    return "Unknown"

In [26]:
for item in medical_terms_found:

    item["section"] = get_section(
        normalized_text,
        item["start"]
    )

print(medical_terms_found)

[{'found': 'patellar tendon rupture', 'normalized': 'patellar tendon rupture', 'start': 37, 'end': 60, 'section': 'Results'}, {'found': 'patellar tendon rupture', 'normalized': 'patellar tendon rupture', 'start': 74, 'end': 97, 'section': 'Impression'}, {'found': 'MRI', 'normalized': 'MRI', 'start': 11, 'end': 14, 'section': 'Technique'}]


In [27]:
NEGATION_TERMS = {
    "no",
    "not",
    "without",
    "absent",
    "absence",
    "intact",
    "preserved"
}

In [28]:
for item in medical_terms_found:

    start = item["start"]

    context = normalized_text[
        max(0, start - 50):start
    ]

    words = context.lower().split()

    item["negation"] = any(
        word in NEGATION_TERMS
        for word in words
    )

print(medical_terms_found)

[{'found': 'patellar tendon rupture', 'normalized': 'patellar tendon rupture', 'start': 37, 'end': 60, 'section': 'Results', 'negation': False}, {'found': 'patellar tendon rupture', 'normalized': 'patellar tendon rupture', 'start': 74, 'end': 97, 'section': 'Impression', 'negation': False}, {'found': 'MRI', 'normalized': 'MRI', 'start': 11, 'end': 14, 'section': 'Technique', 'negation': False}]


In [29]:
doc = nlp(text_for_scispacy)